# Training_File_Optimizer_2x2.ipynb -- F2 Final: sweep optimizer su grid 2x2

**Branch git**: `Training_Method_Exploration` (post-cleanup, audit utente).

**Contesto**: il grid 2x2 (`document/EVENTPROP_GRID2X2.md`) ha mostrato che con AdamW lr=2e-3 fisso, BPTT e EventProp convergono allo stesso val_data plateau entro 1% relativo, su ENTRAMBE le architetture (ALIF e LIF). Quindi EventProp NON migliora BPTT su val_data.

**Obiettivo**: prova DEFINITIVA che il pareggio non e' dovuto a una scelta sub-ottimale dell'optimizer per EventProp. Sweep di 11 configurazioni optimizer x 4 training methods = 44 run. Se anche il MIGLIOR optimizer per EventProp non scende sotto baseline, la chiusura e' rigorosa.

**Configurazioni optimizer (11)**:

| Label | Optimizer | lr | d_coef (Prodigy only) |
|---|---|---|---|
| adamw_lr5e-4 | AdamW | 5e-4 | - |
| adamw_lr1e-3 | AdamW | 1e-3 | - |
| adamw_lr2e-3 | AdamW | 2e-3 | - |
| adamw_lr5e-3 | AdamW | 5e-3 | - |
| adam_lr2e-3 | Adam (no wd) | 2e-3 | - |
| lion_lr1e-4 | Lion | 1e-4 | - |
| prodigy_lr1_d10 | Prodigy | 1.0 | 1.0 (canonical) |
| prodigy_lr1_d05 | Prodigy | 1.0 | 0.5 (mild brake) |
| prodigy_lr1_d03 | Prodigy | 1.0 | 0.3 (med brake) |
| prodigy_lr1_d01 | Prodigy | 1.0 | 0.1 (strong brake) |
| prodigy_lr01_d10 | Prodigy | 0.1 | 1.0 (low init) |

**Training methods (4)**:
- `baseline` -- ALIF + BPTT + surrogate (production)
- `bptt_lif_simple` -- LIF + BPTT + surrogate
- `eventprop_lif_simple` -- LIF + EventProp
- `eventprop_alif_full` -- ALIF + EventProp (replica A1 con EventProp adjoint)

**Setup IDENTICO al grid 2x2** (5 ep, scheduler=none, lambda_data=1.0 puro, po2=1, no OU).

**Tempi stimati Azure CPU** (assumendo ~2x faster del locale):
- baseline (ALIF+BPTT): ~9 min/run
- eventprop_alif_full: ~7 min/run
- LIF (entrambi): ~20s/run
- TOTALE: 4 x 11 = 44 run -> ~3h

**SKIP_IF_EXISTS** attivo: rilancio sicuro dopo interruzioni.

**Esito atteso**:
- Se nessuna config EventProp scende sotto baseline best (~0.222) -> EventProp DEFINITIVAMENTE non e' la cura. Tornare a baseline production.
- Se qualche config EventProp scende sotto baseline best -> rivisitare strategia (improbabile vista evidenza grid 2x2).

In [ ]:
# ===========================================================
# CELLA 0 -- Bootstrap deps + git checkout Training_Method_Exploration
# ===========================================================
import sys

print('Installing/upgrading dependencies (idempotent)...')
!{sys.executable} -m pip install --quiet matplotlib pandas
!{sys.executable} -m pip install --quiet nbstripout
!{sys.executable} -m pip install --quiet prodigyopt   # per --optimizer prodigy

# nbstripout per evitare conflitti git sui notebook (output stripping)
!nbstripout --install --attributes .gitattributes 2>/dev/null
print('   [OK] deps installed + nbstripout attivato')

# Sync con remote
print('\ngit fetch + checkout Training_Method_Exploration + pull:')
!git fetch origin
!git checkout Training_Method_Exploration 2>/dev/null || git checkout -b Training_Method_Exploration origin/Training_Method_Exploration
!git pull --no-rebase --no-edit origin Training_Method_Exploration

print('\nBranch + commit attuale:')
!git branch --show-current
!git log --oneline -3

In [ ]:
# ===========================================================
# CELLA 1 -- ENV CHECK + verifica --training_method CLI
# ===========================================================
import sys, platform, os, subprocess

print(f'Python:          {sys.version.split()[0]} ({platform.system()})')
print(f'Working dir:     {os.getcwd()}')

checks = []
for mod in ['torch', 'numpy', 'pandas', 'matplotlib', 'prodigyopt']:
    try:
        m = __import__(mod)
        v = getattr(m, '__version__', '?')
        print(f'   [OK] {mod:<14} v{v}')
        checks.append((mod, True))
    except Exception as e:
        print(f'   [FAIL] {mod:<14} {e}')
        checks.append((mod, False))

import torch
if torch.cuda.is_available():
    print(f'\nCUDA:            [OK] {torch.cuda.get_device_name(0)}')
else:
    print(f'\nCUDA:            [INFO] training su CPU')

print('\nFile critici:')
for f in ['train.py', 'config.py', 'core/network.py', 'core/eventprop.py',
          'core/neurons.py', 'core/hardware.py', 'data/generator.py']:
    exists = os.path.isfile(f)
    print(f'   {"[OK]" if exists else "[MISSING]"} {f}')
    checks.append((f, exists))

# Verifica nuovi CLI flags (--training_method e --prodigy_d_coef)
res = subprocess.run([sys.executable, 'train.py', '--help'], capture_output=True, text=True)
help_text = res.stdout + res.stderr
for flag in ['--training_method', '--prodigy_d_coef', '--noise_scale', '--po2_enabled', '--max_steps_per_epoch']:
    ok = flag in help_text
    print(f'   {"[OK]" if ok else "[MISSING]"} train.py supporta {flag}')
    checks.append((flag, ok))

# Verifica build_model con 4 varianti
try:
    from core.network import build_model
    METHODS_CHECK = ['baseline', 'bptt_lif_simple',
                     'eventprop_lif_simple', 'eventprop_alif_full']
    for v in METHODS_CHECK:
        m = build_model(v)
        n = sum(p.numel() for p in m.parameters())
        print(f'   [OK] build_model({v:25s}) params={n:>5,d}  class={type(m).__name__}')
    checks.append(('build_model 4 variants', True))
except Exception as e:
    print(f'   [FAIL] build_model: {e}')
    checks.append(('build_model 4 variants', False))

n_fail = sum(1 for _, ok in checks if not ok)
if n_fail == 0:
    print('\n[OK] ENV CHECK PASSED -- procedi con Cella 2')
else:
    print(f'\n[FAIL] ENV CHECK FAILED -- {n_fail} problemi')
    raise SystemExit('Env check failed')

In [ ]:
# ===========================================================
# CELLA 2 -- SWEEP_PLAN: 4 methods x 11 optimizer configs = 44 run
# ===========================================================
# Setup IDENTICO al grid 2x2 (vedi document/EVENTPROP_GRID2X2.md):
#  - 5 ep x 190 step, batch=8 val_batch=64 seq=50, scheduler=none
#  - scenario=highway cut=0, h=32 r=8, noise_scale=0 (cache F2), po2=1
#  - lambda_data=1.0 puro (no PINN multi-obj per misura pulita)
#  - max_inf_streak=99999, early_stop=0

import time, subprocess, sys, os, datetime, shutil, glob
import pandas as pd, numpy as np

SWEEP_COMMON = {
    'epochs':               5,
    'max_steps_per_epoch':  190,
    'batch_size':           8,
    'val_batch_size':       64,
    'seq_len':              50,
    'scheduler':            'none',
    'scenario_mix':         'highway',
    'cut_in_ratio':         0.0,
    'cf_hidden_size':       32,
    'cf_rank':              8,
    'noise_scale':          0.0,
    'po2_enabled':          1,
    'lambda_data':          1.0,
    'lambda_phys':          0.0,
    'lambda_ou':            0.0,
    'lambda_bc':            0.0,
    'lambda_sr':            0.0,
    'n_train':              1500,
    'n_val':                300,
    'max_inf_streak':       99999,
    'early_stop_patience':  0,
}

# 4 training methods (riga del grid 2x2)
TRAINING_METHODS = [
    'baseline',             # ALIF + BPTT + surrogate (production)
    'bptt_lif_simple',      # LIF + BPTT + surrogate
    'eventprop_lif_simple', # LIF + EventProp
    'eventprop_alif_full',  # ALIF + EventProp (la TUA arch con EventProp)
]

# 11 optimizer configs (colonna del grid 2x2)
# Format: (label, optimizer, lr, prodigy_d_coef)
OPTIMIZER_CONFIGS = [
    ('adamw_lr5e-4',     'adamw',   5e-4, 1.0),
    ('adamw_lr1e-3',     'adamw',   1e-3, 1.0),
    ('adamw_lr2e-3',     'adamw',   2e-3, 1.0),    # reference (grid 2x2 default)
    ('adamw_lr5e-3',     'adamw',   5e-3, 1.0),
    ('adam_lr2e-3',      'adam',    2e-3, 1.0),
    ('lion_lr1e-4',      'lion',    1e-4, 1.0),
    ('prodigy_lr1_d10',  'prodigy', 1.0,  1.0),
    ('prodigy_lr1_d05',  'prodigy', 1.0,  0.5),
    ('prodigy_lr1_d03',  'prodigy', 1.0,  0.3),
    ('prodigy_lr1_d01',  'prodigy', 1.0,  0.1),
    ('prodigy_lr01_d10', 'prodigy', 0.1,  1.0),
]

# Quali eseguire? Modifica per disabilitare singoli plan.
RUN_METHODS = set(TRAINING_METHODS)
RUN_OPT_LABELS = set(c[0] for c in OPTIMIZER_CONFIGS)

CACHE_PATH = f'data/cache_{SWEEP_COMMON["n_train"]}_{SWEEP_COMMON["scenario_mix"]}_cut{SWEEP_COMMON["cut_in_ratio"]}_ou{SWEEP_COMMON["noise_scale"]}.pt'
cache_exists = '[esistente]' if os.path.isfile(CACHE_PATH) else '[da generare]'

total = len(RUN_METHODS) * len(RUN_OPT_LABELS)
print(f'SWEEP OPTIMIZER 2x2: {len(RUN_METHODS)} methods x {len(RUN_OPT_LABELS)} configs = {total} run')
print(f'Cache F2: {CACHE_PATH} {cache_exists}')
print(f'Setup: 5 ep x 190 step, scheduler=none, lambda_data=1.0 puro, po2=1, noise=0')
print(f'Safety: max_inf_streak=99999, early_stop=0 (no abort)')

# Stima tempo (Azure CPU, ~2x faster del locale)
time_est_per_run = {
    'baseline':              9*60,    # ~9 min
    'bptt_lif_simple':       20,
    'eventprop_lif_simple':  25,
    'eventprop_alif_full':   7*60,    # ~7 min
}
total_seconds = sum(time_est_per_run.get(m, 600) * len(RUN_OPT_LABELS)
                    for m in RUN_METHODS)
print(f'\nStima tempo Azure: {total_seconds/3600:.1f} h ({total_seconds/60:.0f} min)')

print(f'\n=== Configs ===')
for label, opt, lr, dc in OPTIMIZER_CONFIGS:
    in_run = '[RUN]' if label in RUN_OPT_LABELS else '[SKIP]'
    extra = f' d_coef={dc}' if opt == 'prodigy' else ''
    print(f'  {in_run} {label:18s} optimizer={opt:8s} lr={lr}{extra}')
print(f'\n=== Methods ===')
for m in TRAINING_METHODS:
    in_run = '[RUN]' if m in RUN_METHODS else '[SKIP]'
    print(f'  {in_run} {m}')

In [ ]:
# ===========================================================
# CELLA 3 -- RUN sweep (continue-on-failure, push per-run)
# ===========================================================
# SKIP_IF_EXISTS=True (default): se results/<tag>/training_log.csv esiste,
# il plan viene SALTATO. Permette di rilanciare la cella dopo interruzioni.
# Per forzare ri-esecuzione: SKIP_IF_EXISTS=False (o cancella results/<tag>/).
SKIP_IF_EXISTS = True

def _build_cli(method, opt_label, optimizer, lr, prodigy_d_coef):
    tag = f'SW_{method}_{opt_label}'
    full = {**SWEEP_COMMON, 'optimizer': optimizer, 'lr': lr,
            'prodigy_d_coef': prodigy_d_coef, 'tag': tag,
            'training_method': method}
    args = [
        sys.executable, 'train.py',
        '--training_method',     full['training_method'],
        '--epochs',              str(full['epochs']),
        '--n_train',             str(full['n_train']),
        '--n_val',               str(full['n_val']),
        '--batch_size',          str(full['batch_size']),
        '--val_batch_size',      str(full['val_batch_size']),
        '--seq_len',             str(full['seq_len']),
        '--scheduler',           full['scheduler'],
        '--lr',                  str(full['lr']),
        '--optimizer',           full['optimizer'],
        '--prodigy_d_coef',      str(full['prodigy_d_coef']),
        '--scenario_mix',        full['scenario_mix'],
        '--cut_in_ratio',        str(full['cut_in_ratio']),
        '--cf_hidden_size',      str(full['cf_hidden_size']),
        '--cf_rank',             str(full['cf_rank']),
        '--lambda_data',         str(full['lambda_data']),
        '--lambda_phys',         str(full['lambda_phys']),
        '--lambda_ou',           str(full['lambda_ou']),
        '--lambda_bc',           str(full['lambda_bc']),
        '--lambda_sr',           str(full['lambda_sr']),
        '--noise_scale',         str(full['noise_scale']),
        '--po2_enabled',         str(full['po2_enabled']),
        '--max_inf_streak',      str(full['max_inf_streak']),
        '--early_stop_patience', str(full['early_stop_patience']),
        '--max_steps_per_epoch', str(full['max_steps_per_epoch']),
        '--data_cache',          CACHE_PATH,
        '--tag',                 tag,
    ]
    return args, tag

def _push_run(method, opt_label, optimizer, lr, prodigy_d_coef):
    tag = f'SW_{method}_{opt_label}'
    src, dst = f'checkpoints/{tag}', f'results/{tag}'
    if not os.path.isdir(src):
        print(f'   [WARN] {src} mancante')
        return False
    if os.path.isdir(dst): shutil.rmtree(dst)
    os.makedirs(f'{dst}/plots', exist_ok=True)
    for f in glob.glob(f'{src}/*.csv') + glob.glob(f'{src}/*.json'):
        shutil.copy2(f, dst)
    for f in glob.glob(f'{src}/plots/*.png'):
        shutil.copy2(f, f'{dst}/plots/')

    val_str = ''
    if os.path.isfile(f'{dst}/training_log.csv'):
        edf = pd.read_csv(f'{dst}/training_log.csv')
        if len(edf) > 0:
            best_ep = int(edf.val_data.idxmin()) + 1
            val_str = f'best val_data={edf.val_data.min():.4f} (E{best_ep}/{len(edf)})'

    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    extra = f' d_coef={prodigy_d_coef}' if optimizer == 'prodigy' else ''
    msg = (
        f'results (F2 OptSweep 2x2): {tag} ({ts})\n\n'
        f'{val_str}\n'
        f'method={method}  optimizer={optimizer}  lr={lr}{extra}\n'
        f'Branch Training_Method_Exploration\n'
    )
    with open('/tmp/sw_msg.txt', 'w', encoding='utf-8') as f:
        f.write(msg)
    !git add {dst}
    !git commit -F /tmp/sw_msg.txt
    !rm /tmp/sw_msg.txt
    !git pull --no-rebase --no-edit origin Training_Method_Exploration
    !git push origin Training_Method_Exploration
    return True

# ── Esecuzione ───────────────────────────────────────────────
sweep_results = []
t_start = time.time()
run_idx = 0
total_runs = sum(1 for m in TRAINING_METHODS if m in RUN_METHODS
                 for c in OPTIMIZER_CONFIGS if c[0] in RUN_OPT_LABELS)

for method in TRAINING_METHODS:
    if method not in RUN_METHODS:
        print(f'\n[SKIP_METHOD] {method} -- non in RUN_METHODS')
        continue
    for label, opt, lr, dc in OPTIMIZER_CONFIGS:
        if label not in RUN_OPT_LABELS:
            continue
        run_idx += 1
        cli, tag = _build_cli(method, label, opt, lr, dc)

        if SKIP_IF_EXISTS and os.path.isfile(f'results/{tag}/training_log.csv'):
            try:
                ep_existing = pd.read_csv(f'results/{tag}/training_log.csv')
                v_existing = ep_existing.val_data.min() if len(ep_existing) else None
                v_str = f'val_data={v_existing:.4f}' if v_existing is not None else 'empty'
            except Exception:
                v_str = 'unreadable'
            print(f'\n[{run_idx}/{total_runs}] [SKIP_EXIST] {tag}: results/ presente ({v_str}).')
            sweep_results.append({'tag': tag, 'method': method, 'opt': label,
                                  'status': 'skipped_existing', 'elapsed': 0.0})
            continue

        print('\n' + '=' * 78)
        print(f'[{run_idx}/{total_runs}] {tag}')
        print(f'   method={method}  optimizer={opt}  lr={lr}  d_coef={dc}')
        print('=' * 78)
        t0 = time.time()
        res = subprocess.run(cli, capture_output=False)
        elapsed = time.time() - t0
        status = 'ok' if res.returncode == 0 else f'fail({res.returncode})'
        elapsed_total = time.time() - t_start
        eta_min = (elapsed_total / run_idx) * (total_runs - run_idx) / 60
        print(f'\n[{run_idx}/{total_runs}] {tag} -> {status} in {elapsed/60:.1f} min   '
              f'total={elapsed_total/60:.0f}min  ETA={eta_min:.0f}min')

        # Push per-run (anche se fail: i log parziali sono utili)
        print(f'\nPush results/{tag}...')
        _push_run(method, label, opt, lr, dc)
        sweep_results.append({'tag': tag, 'method': method, 'opt': label,
                              'status': status, 'elapsed': elapsed})

print(f'\n{"="*78}\nSWEEP COMPLETATO in {(time.time()-t_start)/60:.0f} min\n{"="*78}')
n_ok = sum(1 for r in sweep_results if r['status'] == 'ok')
n_fail = sum(1 for r in sweep_results if r['status'].startswith('fail'))
n_skip = sum(1 for r in sweep_results if r['status'] == 'skipped_existing')
print(f'OK: {n_ok}  FAIL: {n_fail}  SKIP: {n_skip}')
for r in sweep_results:
    if r['status'] != 'skipped_existing':
        print(f"   {r['tag']:50s} {r['status']:15s} {r['elapsed']/60:.1f}min")

In [ ]:
# ===========================================================
# CELLA 4 -- ANALISI FINALE: tabella 4x11 val_data + verdetto
# ===========================================================
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Carica tutti i risultati
data_rows = []
for method in TRAINING_METHODS:
    for label, opt, lr, dc in OPTIMIZER_CONFIGS:
        tag = f'SW_{method}_{label}'
        p = f'results/{tag}/training_log.csv'
        if not os.path.isfile(p):
            val_best = None; sr_last = None; n_ep = 0
        else:
            try:
                ep = pd.read_csv(p)
                val_best = ep.val_data.min() if len(ep) else None
                sr_last = ep.spike_rate.iloc[-1] if len(ep) else None
                n_ep = len(ep)
            except Exception:
                val_best = None; sr_last = None; n_ep = 0
        data_rows.append({'method': method, 'opt_label': label,
                          'val_data_best': val_best, 'spike_rate_last': sr_last,
                          'n_ep': n_ep})
df = pd.DataFrame(data_rows)

# ── Tabella pivot: val_data (rows=method, cols=optimizer) ──────────────
pivot_val = df.pivot(index='method', columns='opt_label', values='val_data_best')
# Ordina colonne nel format OPTIMIZER_CONFIGS
col_order = [c[0] for c in OPTIMIZER_CONFIGS]
pivot_val = pivot_val[col_order]
# Ordina righe nel format TRAINING_METHODS
pivot_val = pivot_val.reindex(TRAINING_METHODS)

display(Markdown('## TABELLA val_data ep_best (RMSE m/s²) — 4 methods × 11 optimizers'))
display(pivot_val.style.format('{:.4f}').background_gradient(cmap='RdYlGn_r', axis=None))

# ── Best per method ────────────────────────────────────────────────────
display(Markdown('## Best optimizer per method'))
for method in TRAINING_METHODS:
    row = df[df.method == method].dropna(subset=['val_data_best'])
    if len(row) == 0:
        print(f'  {method:25s}  NO DATA')
        continue
    best = row.loc[row.val_data_best.idxmin()]
    worst = row.loc[row.val_data_best.idxmax()]
    print(f'  {method:25s}  best: {best.val_data_best:.4f} ({best.opt_label:18s})  '
          f'worst: {worst.val_data_best:.4f} ({worst.opt_label})')

# ── Tabella pivot: spike_rate ──────────────────────────────────────────
pivot_sr = df.pivot(index='method', columns='opt_label', values='spike_rate_last')
pivot_sr = pivot_sr[col_order].reindex(TRAINING_METHODS)

display(Markdown('## TABELLA spike_rate ep_last (impatto deploy FPGA)'))
display(pivot_sr.style.format('{:.3f}').background_gradient(cmap='RdYlGn_r', axis=None))

# ── Plot: val_data BEST per method (overlay) ───────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(col_order))
colors = plt.cm.tab10(np.linspace(0, 1, len(TRAINING_METHODS)))
for i, method in enumerate(TRAINING_METHODS):
    vals = pivot_val.loc[method].values
    ax.plot(x, vals, 'o-', linewidth=2, markersize=8,
            color=colors[i], label=method)
ax.set_xticks(x); ax.set_xticklabels(col_order, rotation=45, ha='right')
ax.set_ylabel('val_data best (RMSE m/s²)')
ax.set_title('Sweep optimizer 2x2 -- val_data per (method, optimizer)')
ax.axhline(0.222, ls='--', color='gray', alpha=0.5, label='baseline ref (grid 2x2)')
ax.grid(alpha=0.3); ax.legend(fontsize=10)
plt.tight_layout()
out_dir = f'opt_plots/sw_{datetime.datetime.now().strftime("%Y%m%d_%H%M")}'
os.makedirs(out_dir, exist_ok=True)
plt.savefig(f'{out_dir}/val_data_per_method.png', dpi=120, bbox_inches='tight')
plt.show()

# ── Verdetto finale ────────────────────────────────────────────────────
display(Markdown('## VERDETTO finale'))
baseline_best = pivot_val.loc['baseline'].min() if 'baseline' in pivot_val.index else None
ep_alif_best = pivot_val.loc['eventprop_alif_full'].min() if 'eventprop_alif_full' in pivot_val.index else None
ep_lif_best = pivot_val.loc['eventprop_lif_simple'].min() if 'eventprop_lif_simple' in pivot_val.index else None
bptt_lif_best = pivot_val.loc['bptt_lif_simple'].min() if 'bptt_lif_simple' in pivot_val.index else None

if baseline_best is not None:
    print(f'baseline (ALIF+BPTT)        best: {baseline_best:.4f}')
if ep_alif_best is not None:
    delta = ep_alif_best - baseline_best if baseline_best else 0
    print(f'eventprop_alif_full         best: {ep_alif_best:.4f}  vs baseline: {delta:+.4f}')
if bptt_lif_best is not None:
    delta = bptt_lif_best - baseline_best if baseline_best else 0
    print(f'bptt_lif_simple             best: {bptt_lif_best:.4f}  vs baseline: {delta:+.4f}')
if ep_lif_best is not None:
    delta = ep_lif_best - baseline_best if baseline_best else 0
    print(f'eventprop_lif_simple        best: {ep_lif_best:.4f}  vs baseline: {delta:+.4f}')

print()
if baseline_best is not None and ep_alif_best is not None:
    if ep_alif_best < baseline_best - 0.005:
        print(f'[BREAKTHROUGH] EventProp ALIF batte baseline di {baseline_best-ep_alif_best:.4f}')
        print(f'   -> EventProp PUOI essere la cura. Rivisitare strategia.')
    elif abs(ep_alif_best - baseline_best) < 0.005:
        print(f'[PAREGGIO] EventProp ALIF ~= baseline (delta < 0.005)')
        print(f'   -> EventProp NON e\' la cura. Tornare a baseline production.')
        print(f'   -> Floor val_data ~0.22 confermato architetturale, immune a training method.')
    else:
        print(f'[BPTT vince] baseline batte EventProp ALIF di {ep_alif_best-baseline_best:.4f}')
        print(f'   -> EventProp PEGGIORA. Baseline production confermato vincitore.')

print(f'\nGrafici: {out_dir}/')
print(f'CSV completo: results/SW_*/training_log.csv')